In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

property_gdf = gpd.read_parquet("../../data/processed/nsw_property/property.parquet")
bushfire_gdf = gpd.read_parquet("../../data/processed/nsw_bushfire/bushfire.parquet")
site_features = gpd.read_parquet("../../data/processed/site_features/property_site_features_zoning_heritage_sample.parquet")

In [2]:
print("Property rows:", len(property_gdf))
print("Bushfire rows:", len(bushfire_gdf))
print("Site rows:", len(site_features))

print("Property CRS:", property_gdf.crs)
print("Bushfire CRS:", bushfire_gdf.crs)
print("Site CRS:", site_features.crs)

Property rows: 4198396
Bushfire rows: 265786
Site rows: 49950
Property CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbrevia

In [3]:
property_main = property_gdf[property_gdf["propertytype"] == 1].copy()
print(len(property_main))

4193334


In [4]:
bushfire_keep = bushfire_gdf[
    [
        "fid",
        "d_category",
        "d_guidelin",
        "category",
        "guideline",
        "geometry",
    ]
].copy()

In [5]:
if property_main.crs != bushfire_keep.crs:
    property_main = property_main.to_crs(bushfire_keep.crs)

In [6]:
sample_rids = set(site_features["RID"].unique())
property_sample = property_main[property_main["RID"].isin(sample_rids)].copy()
print(len(property_sample))

49950


In [7]:
joined = gpd.sjoin(
    property_sample,
    bushfire_keep,
    how="left",
    predicate="intersects"
)

In [8]:
print("Joined rows:", len(joined))
joined[
    ["RID", "address", "d_category", "d_guidelin"]
].head(10)

Joined rows: 60630


,RID,address,d_category,d_guidelin
173,260,44 NORTHCOTE STREET ABERDARE,NaN,NaN
217,319,10 MEREWETHER CLOSE NORTH ROTHBURY,Vegetation Category 1,v5b
217,319,10 MEREWETHER CLOSE NORTH ROTHBURY,Vegetation Buffer,v5b
217,319,10 MEREWETHER CLOSE NORTH ROTHBURY,Vegetation Category 3,v5b
221,327,112 MATHIESON STREET BELLBIRD HEIGHTS,NaN,NaN
232,338,584 WOLLOMBI ROAD BELLBIRD,Vegetation Buffer,v5b
251,359,41 RUSSELL STREET BRANXTON,NaN,NaN
285,400,14 FOSTER STREET CESSNOCK,NaN,NaN
327,456,13 WILLIAM STREET CESSNOCK,NaN,NaN
415,606,761 WATAGAN CREEK ROAD WATAGAN,Vegetation Category 3,v5b


In [9]:
joined["d_category"].isna().mean()

np.float64(0.6428830611908296)

In [10]:
joined["d_category"].value_counts(dropna=False).head(20)

d_category
NaN                      38978
Vegetation Buffer         8777
Vegetation Category 1     6184
Vegetation Category 3     4250
Vegetation Category 2     2441
Name: count, dtype: int64

In [11]:
joined["RID"].duplicated().mean()

np.float64(0.17615042058386937)

In [12]:
risk_map = {
    "Vegetation Category 1": 3,
    "Vegetation Category 2": 2,
    "Vegetation Category 3": 1,
    "Vegetation Buffer": 1,
}

def collapse_bushfire_group(group: pd.DataFrame) -> pd.Series:
    cat_nonnull = group["d_category"].dropna()
    guide_nonnull = group["d_guidelin"].dropna()

    categories = sorted(cat_nonnull.astype(str).unique().tolist()) if len(cat_nonnull) else []
    guides = sorted(guide_nonnull.astype(str).unique().tolist()) if len(guide_nonnull) else []

    if len(cat_nonnull):
        max_risk = max(risk_map.get(str(x), 0) for x in cat_nonnull)
        primary_cat = cat_nonnull.value_counts().index[0]
    else:
        max_risk = 0
        primary_cat = pd.NA

    return pd.Series(
        {
            "bushfire_flag": int(group["d_category"].notna().any()),
            "bushfire_hit_count": int(group["d_category"].notna().sum()),
            "bushfire_category_count": int(cat_nonnull.nunique()),
            "bushfire_categories": "|".join(categories),
            "bushfire_guidelines": "|".join(guides),
            "primary_bushfire_category": primary_cat,
            "bushfire_risk_level": int(max_risk),
            "has_bushfire_cat1": int("Vegetation Category 1" in categories),
            "has_bushfire_buffer": int("Vegetation Buffer" in categories),
        }
    )

In [13]:
bushfire_features = (
    joined.groupby("RID", as_index=False)
    .apply(collapse_bushfire_group)
    .reset_index(drop=True)
)

print(len(bushfire_features))
bushfire_features.head()

49950


/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_41072/2052294150.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_bushfire_group)


,RID,bushfire_flag,bushfire_hit_count,bushfire_category_count,bushfire_categories,bushfire_guidelines,primary_bushfire_category,bushfire_risk_level,has_bushfire_cat1,has_bushfire_buffer
0,260,0,0,0,,,<NA>,0,0,0
1,319,1,3,3,Vegetation Buffer|Vegetation Category 1|Vegeta...,v5b,Vegetation Category 1,3,1,1
2,327,0,0,0,,,<NA>,0,0,0
3,338,1,1,1,Vegetation Buffer,v5b,Vegetation Buffer,1,0,1
4,359,0,0,0,,,<NA>,0,0,0


In [14]:
site_with_bushfire = site_features.merge(
    bushfire_features,
    on="RID",
    how="left",
)

print(len(site_with_bushfire))
site_with_bushfire.head()

49950


,RID,gurasid,propertytype,valnetpropertystatus,valnetpropertytype,dissolveparcelcount,valnetlotcount,propid,superlot,address,...,heritage_max_significance,bushfire_flag,bushfire_hit_count,bushfire_category_count,bushfire_categories,bushfire_guidelines,primary_bushfire_category,bushfire_risk_level,has_bushfire_cat1,has_bushfire_buffer
0,260,49494103,1,2.0,2.0,1,1,1341,N,44 NORTHCOTE STREET ABERDARE,...,None,0,0,0,,,<NA>,0,0,0
1,319,49494159,1,2.0,2.0,1,1,3169,N,10 MEREWETHER CLOSE NORTH ROTHBURY,...,None,1,3,3,Vegetation Buffer|Vegetation Category 1|Vegeta...,v5b,Vegetation Category 1,3,1,1
2,327,49494166,1,2.0,2.0,1,1,3682,N,112 MATHIESON STREET BELLBIRD HEIGHTS,...,None,0,0,0,,,<NA>,0,0,0
3,338,49494176,1,2.0,2.0,1,1,3958,N,584 WOLLOMBI ROAD BELLBIRD,...,None,1,1,1,Vegetation Buffer,v5b,Vegetation Buffer,1,0,1
4,359,49494197,1,2.0,2.0,1,1,4739,N,41 RUSSELL STREET BRANXTON,...,None,0,0,0,,,<NA>,0,0,0


In [15]:
fill_zero_cols = [
    "bushfire_flag",
    "bushfire_hit_count",
    "bushfire_category_count",
    "bushfire_risk_level",
    "has_bushfire_cat1",
    "has_bushfire_buffer",
]

for col in fill_zero_cols:
    site_with_bushfire[col] = site_with_bushfire[col].fillna(0).astype(int)

In [16]:
print(site_with_bushfire["bushfire_flag"].value_counts(normalize=True, dropna=False))
print(site_with_bushfire["primary_bushfire_category"].value_counts(dropna=False))
print(site_with_bushfire["bushfire_risk_level"].value_counts(dropna=False))

bushfire_flag
0    0.78034
1    0.21966
Name: proportion, dtype: float64
primary_bushfire_category
<NA>                     38978
Vegetation Buffer         6137
Vegetation Category 3     2290
Vegetation Category 1     2124
Vegetation Category 2      421
Name: count, dtype: int64
bushfire_risk_level
0    38978
1     6735
3     3795
2      442
Name: count, dtype: int64


In [18]:
Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

site_with_bushfire.to_parquet(
    "../../data/processed/site_features/property_site_features_zoning_heritage_bushfire_sample.parquet",
    index=False,
)